# Flagship capstone: synthetic RF detection

Connect mathematical detection theory to measured ML performance, reproducible artifacts and deployment. This is a synthetic teaching experiment, not an operational radar qualification.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


## 1. Define the task and assumptions

Decide between H0: x=n and H1: x=s+n using 128 complex-IQ samples. Noise has expected power E|n|²=1. SNR is mean in-window signal power relative to that noise power. Derive the known-signal Gaussian likelihood-ratio statistic and explain the additional unknown-frequency search.

Data roles are training, validation, probability calibration, threshold selection and untouched testing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from capstones.rf_detection.data import generate
from capstones.rf_detection.inference import features, baseline_scores
batch=generate(10,SEED,0)
print("IQ shape",batch.iq.shape,"features",features(batch.iq).shape)
plt.plot(batch.iq[1].real,label="I")
plt.plot(batch.iq[1].imag,label="Q")
plt.legend();plt.title("One synthetic recording");plt.show()

## 2. Predeclare the experiment

Five model families, three seeds, four conditions and six SNR values. The feature model is the fixed serving reference; do not pick its seed using final test results. Smaller settings exercise the code but do not reproduce a larger run.

In [ ]:
from capstones.rf_detection.train import run
OUTPUT=Path(os.environ.get('COURSE_RESULTS_DIR', ROOT/'results'))/'rf-notebook'
SEEDS=(0,1,2)
metrics=run(OUTPUT,seeds=SEEDS,deep=True,n_train=600,n_eval=400,epochs=6,pfa=.01)
print('Measured combinations:',len(metrics))
display(metrics.head())

## 3. Compare baselines and neural models

Thresholds are frozen across conditions. The matched-filter bank may outperform a learned model; that is an informative outcome, not an experiment failure. Per-seed Wilson intervals quantify finite test counts; across-seed variability has a different meaning.

In [ ]:
subset=metrics.query('condition == "in_distribution"')
for name, rows in subset.groupby('model'):
    averages=rows.groupby('snr_db').pd.mean()
    plt.plot(averages.index,averages.values,marker='o',label=name)
plt.ylim(0,1);plt.xlabel('SNR (dB)');plt.ylabel('Pd');plt.legend();plt.show()
display(metrics.groupby(['model','condition'])[['pd','pfa','brier']].mean())

## 4. Prove serialization and API parity

The API loads a checksummed JSON artifact, not executable pickle data. This in-process check is distinct from the real Docker/HTTP parity test run in CI.

In [ ]:
from capstones.rf_detection.inference import load_artifact,predict
from capstones.rf_detection.app import create_app
from fastapi.testclient import TestClient
payload=load_artifact(OUTPUT/'model.json')
iq=generate(2,SEED,900).iq[0]
expected,_=predict(payload,iq)
request={'iq':np.column_stack([iq.real,iq.imag]).tolist()}
with TestClient(create_app(OUTPUT/'model.json')) as client:
    response=client.post('/predict',json=request)
    assert response.status_code==200
    assert abs(response.json()['probability']-expected[0])<1e-12
    print(response.json())

## 5. Observe shift without pretending to know accuracy

Feature drift is not labeled performance. Investigate provenance, receiver configuration and sampling before deciding to retrain.

In [ ]:
from capstones.rf_detection.monitoring import drift_report
reference=generate(400,SEED,910)
live=generate(400,SEED,911,condition='colored_noise')
report=drift_report(reference.iq,live.iq)
display(pd.DataFrame(report['features']))
print(report['performance'],report['action'])

## 6. Deploy and defend

Use the [README](README.md) to build and run the Docker image. The notebook does not silently install Docker or launch a public service.

Explain why each split exists, which assumptions support the threshold, the observed neural-versus-baseline tradeoff, false-alarm uncertainty, prevalence dependence, latency scope, artifact checks and rollback criteria. Complete Exam 06 only after you can defend the whole pipeline without reading the implementation.